# Posterior ensemble

## Experiment output folder

In [ ]:
from pathlib import Path

# Define the subfolder path for outputs
output_dir = Path("output/ETNA2018")

## Load the prior and posterior (latent space)

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

prior          = xr.open_dataset(output_dir / "prior.nc")
posterior_etkf = xr.open_dataset(output_dir / "posterior-latent-etkf.nc")
posterior_pf   = xr.open_dataset(output_dir / "posterior-latent-pf.nc")

nens    = prior.sizes['ens']
nlatent = prior.sizes['latent_dim']

Z_prior         = prior['z'].to_numpy()
Z_analysis_etkf = posterior_etkf['z'].to_numpy()
Z_analysis_pf   = posterior_pf['z'].to_numpy()

In [ ]:
nlatent, nens

## Statistics in latent space

In [ ]:
spread_z = {
    "prior": Z_prior.std(axis=0, ddof=1).mean(),
    "etkf":  Z_analysis_etkf.std(axis=0, ddof=1).mean(),
    "pf":    Z_analysis_pf.std(axis=0, ddof=1).mean(),
}
spread_z

In [ ]:
variance_z = {
    "prior": Z_prior.var(axis=0, ddof=1).sum(),
    "etkf":  Z_analysis_etkf.var(axis=0, ddof=1).sum(),
    "pf":    Z_analysis_pf.var(axis=0, ddof=1).sum(),
}
variance_z

In [ ]:
df = pd.DataFrame({
    "prior": Z_prior.var(axis=0, ddof=1),
    "etkf":  Z_analysis_etkf.var(axis=0, ddof=1),
    "pf":    Z_analysis_pf.var(axis=0, ddof=1),
})

df.sum()

In [ ]:
variance_sorted = df.sort_values("etkf", ascending=True)
variance_sorted.head(8)

## Visualization in the latent space

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

In [ ]:
# ------------------------------------------------------------
# Publication settings
# ------------------------------------------------------------
 
sns.set_theme(
    style="white",
    context="paper",
)

# ------------------------------------------------------------
# Category / color configuration
# ------------------------------------------------------------
 
categories = {
    "prior": {
        "color_line": "#62748E",
        "color_face": "#FFFFFF",
        "alpha": 0.5,
        "histtype": "step",
        "label": "Prior",
    },
    "etkf": {
        "color_line": "#ff7f0e",
        "color_face": "#ffbb78",
        "alpha": 0.35,
        "histtype": "stepfilled",
        "label": "ETKF Posterior",
    },
    "pf": {
        "color_line": "#2ca02c",
        "color_face": "#98df8a",
        "alpha": 0.35,
        "histtype": "stepfilled",
        "label": "PF Posterior",
    },
}

# Map each category to the data array it corresponds to.
data_map = {
    "prior": Z_prior,
    "etkf":  Z_analysis_etkf,
    "pf":    Z_analysis_pf,
}

pairs = [(15,19),(18,11),(24,8)]
npairs = len(pairs)

In [ ]:
fig = plt.figure(
    figsize=(3, 9),
    dpi=200,
)

outer = fig.add_gridspec(
    npairs, 1,
    wspace=0.10,
    hspace=0.15,
)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

for k, (i, j) in enumerate(pairs):

    #row = k // 3
    #col = k % 3
    row = k
    col = 0

    inner = outer[row, col].subgridspec(
        2, 2,

        # Bigger marginals
        width_ratios=[3.5, 2.0],
        height_ratios=[2.0, 3.5],

        wspace=0.02,
        hspace=0.02,
    )

    ax = fig.add_subplot(inner[1, 0])

    ax_top = fig.add_subplot(
        inner[0, 0],
        sharex=ax,
    )

    ax_right = fig.add_subplot(
        inner[1, 1],
        sharey=ax,
    )

    # --------------------------------------------------------
    # Main scatter
    # --------------------------------------------------------

    for cat_key, cat in categories.items():
        data = data_map[cat_key]
        ax.scatter(
            data[:, j],
            data[:, i],
            s=7,
            linewidth=0.3,
            alpha=cat["alpha"],
            facecolor=cat["color_face"],
            edgecolor=cat["color_line"],
            label=cat["label"],
        )

    # --------------------------------------------------------
    # Common histogram ranges
    # --------------------------------------------------------

    x_bins = np.linspace(
        Z_prior[:, j].min(),
        Z_prior[:, j].max(),
        40,
    )

    y_bins = np.linspace(
        Z_prior[:, i].min(),
        Z_prior[:, i].max(),
        40,
    )

    # --------------------------------------------------------
    # Top histogram
    # --------------------------------------------------------

    for cat_key, cat in categories.items():
        data = data_map[cat_key]
        ax_top.hist(
            data[:, j],
            bins=x_bins,
            density=True,
            histtype=cat["histtype"],
            facecolor=cat["color_face"],
            edgecolor=cat["color_line"],
            alpha=cat["alpha"],
            linewidth=1.2,
        )

    # --------------------------------------------------------
    # Right histogram
    # --------------------------------------------------------

    for cat_key, cat in categories.items():
        data = data_map[cat_key]
        ax_right.hist(
            data[:, i],
            bins=y_bins,
            density=True,
            orientation="horizontal",
            histtype=cat["histtype"],
            facecolor=cat["color_face"],
            edgecolor=cat["color_line"],
            alpha=cat["alpha"],
            linewidth=1.2,
        )

    # --------------------------------------------------------
    # Remove histogram axes
    # --------------------------------------------------------

    ax_top.tick_params(
        left=False,
        bottom=False,
        labelleft=False,
        labelbottom=False,
    )

    ax_right.tick_params(
        left=False,
        bottom=False,
        labelleft=False,
        labelbottom=False,
    )

    ax_top.set_ylabel("")
    ax_top.set_xlabel("")

    ax_right.set_ylabel("")
    ax_right.set_xlabel("")

    # --------------------------------------------------------
    # Main axes
    # --------------------------------------------------------

    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.tick_params(
        labelsize=8,
        direction="out",
        length=3,
    )

    # --------------------------------------------------------
    # Pair label
    # --------------------------------------------------------

    ax.text(
        0.04,
        0.94,
        rf"$Z_{{{i}}}$ vs $Z_{{{j}}}$",
        transform=ax.transAxes,
        fontsize=10,
        va="top",
    )

    # --------------------------------------------------------
    # Panel label
    # --------------------------------------------------------

    ax.text(
        0.04,
        0.06,
        f"({chr(97 + k)})",
        transform=ax.transAxes,
        fontsize=10,
        #fontweight="bold",
    )

    # --------------------------------------------------------
    # Clean spines
    # --------------------------------------------------------

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax_top.spines["top"].set_visible(False)
    ax_top.spines["right"].set_visible(False)
    ax_top.spines["left"].set_visible(False)
    ax_top.spines["bottom"].set_visible(False)

    ax_right.spines["top"].set_visible(False)
    ax_right.spines["right"].set_visible(False)
    ax_right.spines["left"].set_visible(False)
    ax_right.spines["bottom"].set_visible(False)

# ------------------------------------------------------------
# Legend
# -----------------------------------------------------------

legend_handles = [
    Patch(
        facecolor=cat["color_face"],
        edgecolor=cat["color_line"],
        label=cat["label"],
    )
    for cat in categories.values()
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.7, 0.9),
    ncol=1,
    frameon=True,
    fontsize=9,
)

## Decode the posterior

#### Model loading

In [ ]:
import torch
from torch.utils.data import DataLoader

from modules.model import VariationalAutoencoder
from modules.dataset import LogMinMaxScale, EnsembleDataset

checkpoint = torch.load(output_dir / 'model.pt', map_location='cpu')

## Normalization
min_value = checkpoint['MINVAL']
max_value = checkpoint['MAXVAL']
scale     = checkpoint['SCALE']
transform = LogMinMaxScale(min_value, max_value, scale)

# Model
model = VariationalAutoencoder(checkpoint['LATENT_DIM'], in_shape=checkpoint['IN_SHAPE'])
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
device = torch.device('cpu')
latent_dim = checkpoint['LATENT_DIM']

#### Decoding

In [ ]:
model.eval()

with torch.no_grad():
    z = torch.from_numpy(Z_analysis).float().to(device)
    x_analysis = model.decode(z)
    x_analysis_raw = transform.invert(x_analysis).squeeze()
    
    # Detach and move to CPU once right after creation
    x_analysis_raw = x_analysis_raw.detach().cpu()

In [ ]:
posterior_samples = xr.Dataset(
    data_vars={
        "samples": (
            ("ens", "lat", "lon"),
            x_analysis_raw.numpy()
        ),
        "mean": (
            ("lat", "lon"),
            x_analysis_raw.mean(dim=0).numpy()
        ),
    },
    coords={
        "ens": np.arange(nens),
        #"lat": lat,
        #"lon": lon,
    },
)

posterior_samples.to_netcdf(output_dir / "posterior-samples-pf.nc")

## Posterior in the physical space

#### Load data

In [ ]:
fname1 = "data/etna2018-128.ens.nc"
fname2 = output_dir / "prior.nc"
fname3 = output_dir / "posterior-samples-etkf.nc"
fname4 = output_dir / "posterior-samples-pf.nc"
fname5 = "data/20181224_1800_Meteosat-11_Etna_VPRoutput.nc"

prior_model    = xr.open_dataset(fname1)
prior_vae      = xr.open_dataset(fname2)
posterior_etkf = xr.open_dataset(fname3)
posterior_pf   = xr.open_dataset(fname4)
observations   = xr.open_dataset(fname5)

In [ ]:
lat1d = prior_model["lat"]
lon1d = prior_model["lon"]

# Get model domain limits
lat_min, lat_max = lat1d.min().item(), lat1d.max().item()
lon_min, lon_max = lon1d.min().item(), lon1d.max().item()

# Caution: Posteriors dont save coordinate variables!!
posterior_etkf = posterior_etkf.assign_coords(lat=lat1d, lon=lon1d)
posterior_pf   = posterior_pf.assign_coords(lat=lat1d, lon=lon1d)

In [ ]:
rescaling_factor = 1E3 # convert to mg/m2

data_map = {
    "prior_model": rescaling_factor * prior_model["tephra_col_mass"],
    "prior":       rescaling_factor * prior_vae["samples"],
    "etkf":        rescaling_factor * posterior_etkf["samples"],
    "pf":          rescaling_factor * posterior_pf["samples"],
}

#### Ensemble spread

In [ ]:
spread_x = {key: da.std(dim="ens", ddof=1).mean().item() for key, da in data_map.items()}
spread_x

In [ ]:
variance_x = {key: da.var(dim="ens", ddof=1).mean().item() for key, da in data_map.items()}
variance_x

#### Get observations

In [ ]:
# Build 2D boolean mask directly on xarray / NumPy
lat2d = observations["latitude"].values
lon2d = observations["longitude"].values

valid = (
    (lat2d >= lat_min) & (lat2d <= lat_max) &
    (lon2d >= lon_min) & (lon2d <= lon_max)
)

# Extract 1D vectors for valid points directly
y_obs   = observations["mass"].values[valid].astype(np.float64)
lat_obs = lat2d[valid].astype(np.float64)
lon_obs = lon2d[valid].astype(np.float64)

# Required for interpolations
lat_points = xr.DataArray(lat_obs, dims="obs")
lon_points = xr.DataArray(lon_obs, dims="obs")

# Rescaling
y_obs *= rescaling_factor

In [ ]:
N_obs = len(y_obs)
print(f"Number of observations: {N_obs}")

#### Agreement with assimilated observations

In [ ]:
from scipy.stats import pearsonr

In [ ]:
threshold = 100.0  # mg/m2

obs_event = y_obs > threshold

rmse = {}
rmsle = {}
pearson = {}
pod = {}
iou = {}

for label, X in data_map.items():

    # Interpolate every ensemble member
    Y_xr = X.interp(
        lat=lat_points,
        lon=lon_points,
        method="linear"
    )

    Y = Y_xr.values.astype(np.float64)   # (ens, n_obs)

    # Metrics for each ensemble member
    error = Y - y_obs[None, :]
    log_error = np.log1p(Y) - np.log1p(y_obs)[None, :]

    rmse_members = np.sqrt(np.mean(error**2, axis=1))
    rmsle_members = np.sqrt(np.mean(log_error**2, axis=1))

    pearson_members = np.array([
        pearsonr(y, y_obs).statistic
        for y in Y
    ])

    # Categorical metrics
    model_event = Y > threshold

    hits = np.sum(
        model_event & obs_event[None, :],
        axis=1
    )

    union = np.sum(
        model_event | obs_event[None, :],
        axis=1
    )

    pod_members = hits / np.sum(obs_event)
    iou_members = hits / union

    # Ensemble-average metrics
    rmse[label]    = np.mean(rmse_members)
    rmsle[label]   = np.mean(rmsle_members)
    pearson[label] = np.mean(pearson_members)
    pod[label]     = np.mean(pod_members)
    iou[label]     = np.mean(iou_members)

print("RMSE:\n", rmse)
print("RMSLE:\n", rmsle)
print("Pearson correlation:\n", pearson)
print("POD:\n", pod)
print("Jaccard:\n", iou)

## Summary

In [ ]:
rmse_metrics = {
    "RMSE": rmse,
    "RMSLE": rmsle,
    "Pearson correlation": pearson,
    "POD": pod,
    "Jaccard index": iou,
}

df_rmse = pd.DataFrame(metrics).T
df_rmse.columns = [
    "Model Prior",
    "VAE Prior",
    "ETKF Posterior",
    "PF Posterior",
]

table1 = df_rmse.to_latex(
    float_format="%.3f",
    escape=False,
)
print(table1)

In [ ]:
spread_metrics = {
    "Latent-space spread": spread_z,
    "Physical-space spread": {k: v for k, v in spread_x.items() if k != "prior_model"},
}

df_spread = pd.DataFrame(spread_metrics)
df_spread.index = ["VAE Prior", "ETKF Posterior", "PF Posterior"]

table2 = df_spread.to_latex(
    float_format="%.3f",
    escape=False,
)
print(table2)